# 🛠️ Setup & Data Generation

Welcome to **SQL Practice — Window Functions for Data Engineers**!

This notebook:
1. Verifies your environment (DuckDB, pandas, etc.)
2. Generates all synthetic datasets used by the other notebooks
3. Shows a quick preview of each table

## Datasets
| Table | Rows | Description |
|-------|------|-------------|
| `employees` | 50 | Staff across 6 departments with salaries & hire dates |
| `products` | 20 | Product catalog with categories & prices |
| `sales` | ~250 | Sales-rep performance records (2022–2023) |
| `orders` | 300 | Customer orders linked to products |


## 1 · Environment Check

In [8]:
import sys
import duckdb
import pandas as pd
import numpy as np

print(f"Python  : {sys.version.split()[0]}")
print(f"DuckDB  : {duckdb.__version__}")
print(f"Pandas  : {pd.__version__}")
print(f"NumPy   : {np.__version__}")


Python  : 3.14.4
DuckDB  : 1.5.4
Pandas  : 3.0.3
NumPy   : 2.5.0


## 2 · Generate Datasets

Run the cell below to create the CSV files in the `data/` folder.

In [9]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "../data/generate_data.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)


Generating datasets...
  employees :   50 rows  -> c:\Users\pedro\Documents\sql_practice\data/employees.csv
  products  :   20 rows  -> c:\Users\pedro\Documents\sql_practice\data/products.csv
  sales     :  285 rows  -> c:\Users\pedro\Documents\sql_practice\data/sales.csv
  orders    :  300 rows  -> c:\Users\pedro\Documents\sql_practice\data/orders.csv
Done.



## 3 · Load Data into DuckDB

DuckDB can query CSV files directly via `read_csv_auto()` or register pandas DataFrames as virtual tables.

In [10]:
import duckdb
import pandas as pd

# Load CSVs
employees = pd.read_csv("../data/employees.csv")
products  = pd.read_csv("../data/products.csv")
sales     = pd.read_csv("../data/sales.csv")
orders    = pd.read_csv("../data/orders.csv")

# Create an in-memory DuckDB connection and register DataFrames
con = duckdb.connect()
con.register("employees", employees)
con.register("products",  products)
con.register("sales",     sales)
con.register("orders",    orders)

print("Tables registered:", con.execute("SHOW TABLES").fetchdf()["name"].tolist())


Tables registered: ['employees', 'orders', 'products', 'sales']


## 4 · Quick Data Preview

In [11]:
# Helper to run SQL and display results
def sql(query, n=10):
    return con.execute(query).fetchdf().head(n)


In [12]:
print("=== employees ===")
display(sql("SELECT * FROM employees LIMIT 5"))

print("\n=== products ===")
display(sql("SELECT * FROM products LIMIT 5"))

print("\n=== sales ===")
display(sql("SELECT * FROM sales LIMIT 5"))

print("\n=== orders ===")
display(sql("SELECT * FROM orders LIMIT 5"))


=== employees ===


,employee_id,first_name,last_name,full_name,department,job_title,salary,hire_date,manager_id,city
0,1,Michael,Lopez,Michael Lopez,Engineering,Junior Engineer,199752,2022-03-04,NaN,Chicago
1,2,Lisa,Martin,Lisa Martin,Engineering,Engineer,133984,2015-04-13,1.0,Phoenix
2,3,Mark,Williams,Mark Williams,Engineering,Senior Engineer,125774,2017-09-30,1.0,Houston
3,4,Joseph,Jones,Joseph Jones,Engineering,Staff Engineer,87568,2016-07-25,1.0,Chicago
4,5,David,Jackson,David Jackson,Engineering,Principal Engineer,205151,2022-08-03,1.0,Dallas



=== products ===


,product_id,product_name,category,unit_price,cost,margin_pct
0,1,Laptop Pro 15,Electronics,1299.99,780.0,40.0
1,2,Wireless Mouse,Electronics,29.99,12.0,60.0
2,3,Standing Desk,Furniture,499.00,220.0,55.9
3,4,Office Chair,Furniture,389.00,155.0,60.2
4,5,Monitor 27in,Electronics,349.99,175.0,50.0



=== sales ===


,sale_id,rep_id,rep_name,region,category,amount,sale_date,year,quarter,month,month_label
0,22,16,Christopher Anderson,Northeast,Training,51712.40,2022-01-01,2022,Q1,1,2022-01
1,115,20,John Anderson,Northeast,Support,45153.39,2022-01-02,2022,Q1,1,2022-01
2,241,26,Ashley Davis,Southwest,Support,35906.96,2022-01-03,2022,Q1,1,2022-01
3,78,19,Betty Davis,West,Support,57432.65,2022-01-09,2022,Q1,1,2022-01
4,72,19,Betty Davis,West,Support,40037.70,2022-01-10,2022,Q1,1,2022-01



=== orders ===


,order_id,customer_id,customer_name,product_id,product_name,category,order_date,year,quarter,month,month_label,quantity,unit_price,total_amount,region
0,78,13,Oceanic Airlines,3,Standing Desk,Furniture,2022-01-03,2022,Q1,1,2022-01,20,499.00,9980.00,Midwest
1,54,13,Oceanic Airlines,11,Desk Lamp LED,Furniture,2022-01-15,2022,Q1,1,2022-01,13,39.99,519.87,Midwest
2,13,9,Sterling Cooper,16,Cloud Storage 1TB/yr,Software,2022-01-16,2022,Q1,1,2022-01,15,99.99,1499.85,Southeast
3,216,10,Vandelay Industries,9,Whiteboard XL,Office Supplies,2022-01-17,2022,Q1,1,2022-01,18,75.00,1350.00,West
4,82,10,Vandelay Industries,13,Pen Set Premium,Office Supplies,2022-01-17,2022,Q1,1,2022-01,11,12.99,142.89,Southeast


## 5 · Schema Summary

In [13]:
for tbl in ["employees", "products", "sales", "orders"]:
    print(f"\n--- {tbl} ---")
    display(con.execute(f"DESCRIBE {tbl}").fetchdf())



--- employees ---


,column_name,column_type,null,key,default,extra
0,employee_id,BIGINT,YES,None,None,None
1,first_name,VARCHAR,YES,None,None,None
2,last_name,VARCHAR,YES,None,None,None
3,full_name,VARCHAR,YES,None,None,None
4,department,VARCHAR,YES,None,None,None
5,job_title,VARCHAR,YES,None,None,None
6,salary,BIGINT,YES,None,None,None
7,hire_date,VARCHAR,YES,None,None,None
8,manager_id,DOUBLE,YES,None,None,None
9,city,VARCHAR,YES,None,None,None



--- products ---


,column_name,column_type,null,key,default,extra
0,product_id,BIGINT,YES,None,None,None
1,product_name,VARCHAR,YES,None,None,None
2,category,VARCHAR,YES,None,None,None
3,unit_price,DOUBLE,YES,None,None,None
4,cost,DOUBLE,YES,None,None,None
5,margin_pct,DOUBLE,YES,None,None,None



--- sales ---


,column_name,column_type,null,key,default,extra
0,sale_id,BIGINT,YES,None,None,None
1,rep_id,BIGINT,YES,None,None,None
2,rep_name,VARCHAR,YES,None,None,None
3,region,VARCHAR,YES,None,None,None
4,category,VARCHAR,YES,None,None,None
5,amount,DOUBLE,YES,None,None,None
6,sale_date,VARCHAR,YES,None,None,None
7,year,BIGINT,YES,None,None,None
8,quarter,VARCHAR,YES,None,None,None
9,month,BIGINT,YES,None,None,None



--- orders ---


,column_name,column_type,null,key,default,extra
0,order_id,BIGINT,YES,None,None,None
1,customer_id,BIGINT,YES,None,None,None
2,customer_name,VARCHAR,YES,None,None,None
3,product_id,BIGINT,YES,None,None,None
4,product_name,VARCHAR,YES,None,None,None
5,category,VARCHAR,YES,None,None,None
6,order_date,VARCHAR,YES,None,None,None
7,year,BIGINT,YES,None,None,None
8,quarter,VARCHAR,YES,None,None,None
9,month,BIGINT,YES,None,None,None


## 6 · Your First Window Function

Before jumping into the exercises, here is the canonical syntax for a window function:

```sql
function_name(column)
    OVER (
        PARTITION BY partition_col   -- optional: group rows
        ORDER BY order_col           -- optional: define row order
        ROWS/RANGE BETWEEN ...       -- optional: define frame
    )
```

Let's see one in action — rank employees by salary within their department:

In [14]:
sql("""
SELECT
    full_name,
    department,
    salary,
    RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dept_salary_rank
FROM employees
ORDER BY department, dept_salary_rank
""")


,full_name,department,salary,dept_salary_rank
0,Michael Moore,Engineering,209375,1
1,David Jackson,Engineering,205151,2
2,Michael Lopez,Engineering,199752,3
3,Daniel Moore,Engineering,172538,4
4,Mark Rodriguez,Engineering,152563,5
5,Linda Miller,Engineering,147592,6
6,Lisa Martin,Engineering,133984,7
7,Margaret Brown,Engineering,133190,8
8,Mark Williams,Engineering,125774,9
9,David Rodriguez,Engineering,124504,10


---
## Next Steps

| Notebook | Topic |
|----------|-------|
| `01_ranking_functions.ipynb` | ROW_NUMBER, RANK, DENSE_RANK, NTILE |
| `02_offset_functions.ipynb` | LAG, LEAD, FIRST_VALUE, LAST_VALUE |
| `03_running_aggregates.ipynb` | SUM/AVG/COUNT OVER, rolling windows |
| `04_advanced_challenges.ipynb` | Real-world data-engineering scenarios |